In [ ]:
import os  
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import rc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
import imageio.v2 as imageio

import metpy.calc as mpcalc
from metpy.units import units

#########################################
# Set up the output directory
#########################################
output_parent = "/home/weiji/restart_exam/20250221/polar_vortex_analysis/PV/base_on_Pressure"
os.makedirs(output_parent, exist_ok=True)

#########################################
# Define years, months and pressure levels
#########################################
years = ['0008', '0013', '0014', '0019']
months = ['Feb', 'Mar']

# Pressure levels in hPa for PV calculation on pressure surfaces
pv_pressure_levels = [10, 50]

#########################################
# Function: load_experiment_field_for_var
#########################################
def load_experiment_field_for_var(file_pattern, var_name, lat_min=60, lat_max=90):
    """
    Load experimental field data for a given variable and latitude range.
    """
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    data = ds[var_name]
    data = data.sel(lat=slice(lat_min, lat_max))
    if 'member' in data.dims:
        data = data.mean(dim='member')
    return data

#########################################
# Functions: Load data for 'nocoupl' and 'small'
#########################################
def load_year_nocouple_field_var(year, month, var_name, lat_min=60, lat_max=90):
    """
    Load field variable for nocoupled data for a specific year and month.
    """
    if month == 'Feb':
        file_pattern = f'/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc'
    elif month == 'Mar':
        file_pattern = f'/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc'
    else:
        raise ValueError("Invalid month")
    return load_experiment_field_for_var(file_pattern, var_name, lat_min, lat_max)

def load_year_small_pert_field_var(year, month, var_name, lat_min=60, lat_max=90):
    """
    Load field variable for small perturbation data for a specific year and month.
    """
    if month == 'Feb':
        file_pattern = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}/Feb/BWCN.e122.f19_g16.002.{year}-02.*.nc'
    elif month == 'Mar':
        file_pattern = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}/Mar/BWCN.e122.f19_g16.002.{year}-03.*.nc'
    else:
        raise ValueError("Invalid month")
    return load_experiment_field_for_var(file_pattern, var_name, lat_min, lat_max)

#########################################
# Function: compute_pv_pressure_full
#########################################
def compute_pv_pressure_full(T_field, U_field, V_field, plev, lat, lon, target_pressure):
    """
    Compute potential vorticity (PV) from a full 3D field on pressure surfaces.
    
    Parameters:
        T_field, U_field, V_field: 3D arrays (nlev, nlat, nlon) with units [K], [m/s], [m/s].
        plev: 1D array of pressure levels in Pa.
        lat, lon: 1D arrays of latitude (in degrees) and longitude.
        target_pressure: target pressure in Pa.
    Returns:
        2D array (lat, lon) of PV in PVU (1 PVU = 1e-6 K*m²/(kg*s)).
    """
    if T_field.ndim < 3:
         raise ValueError("Input T_field must be 3-dimensional")
    
    # Reshape plev for broadcasting
    plev_reshaped = plev[:, None, None]
    
    # Calculate potential temperature theta
    theta = mpcalc.potential_temperature(plev_reshaped * units.Pa, T_field * units.K)
    
    # Compute grid spacing
    dlon = np.diff(lon).mean()
    dlat = np.diff(lat).mean()
    R = 6371000  # Earth's radius in meters
    lat_rad = np.deg2rad(lat)

    # Compute dx (in meters)
    dx_meters = dlon * np.cos(lat_rad[:, np.newaxis]) * (2 * np.pi * R) / 360
    dx = np.repeat(dx_meters, nlon, axis=1)  # shape (nlat, nlon)

    # Compute dy (in meters)
    dy_meters = dlat * (2 * np.pi * R) / 360
    dy = np.full((nlat, nlon), dy_meters)  # shape (nlat, nlon)

    # Now, pass dx and dy to the PV calculation
    pv_field = mpcalc.potential_vorticity_baroclinic(
        theta, 
        plev * units.Pa,
        U_field * units("m/s"),
        V_field * units("m/s"),
        dx * units.m,
        dy * units.m,
        latitude=lat2d_rad,
        x_dim=2,
        y_dim=1,
        vertical_dim=0
    )
    
    
    # If pv_field is 3D, select the layer closest to target_pressure; otherwise return pv_field directly
    if pv_field.ndim == 3:
         idx = np.argmin(np.abs(plev - target_pressure))
         pv_target = pv_field[idx, :, :]
    else:
         pv_target = pv_field
    
    # Convert to base units and then to PVU (1 PVU = 1e-6 K*m²/(kg*s))
    pv_converted = pv_target.to('K*m**2/(kg*s)').magnitude / 1e-6
    return pv_converted

#########################################
# Function: extend_to_pole
#########################################
def extend_to_pole(pv_da):
    """
    Extend the PV DataArray to include the pole (90°) if not present.
    It duplicates the last latitudinal row (nearest neighbor) to create a new grid point at 90°.
    """
    current_lat = pv_da.lat.values
    if current_lat[-1] < 90:
        new_lat = np.append(current_lat, 90.0)
        # Duplicate the last row (axis=1 corresponds to latitude dimension)
        new_values = np.concatenate([pv_da.values, pv_da.values[:, -1:, :]], axis=1)
        pv_da_extended = xr.DataArray(new_values, coords=[pv_da.time, new_lat, pv_da.lon], dims=['time', 'lat', 'lon'])
        return pv_da_extended
    else:
        return pv_da

#########################################
# Function: plot_composite_daily_maps_PV
#########################################
def plot_composite_daily_maps_PV(exp1_name, pv_data1, exp2_name, pv_data2, output_dir, label, frame_duration=500):
    os.makedirs(output_dir, exist_ok=True)
    rc('font', **{'family': 'sans-serif', 'sans-serif': ['Helvetica']})
    rc('text', usetex=False)
    
    # 固定色标范围设置：对于10 hPa和50 hPa分别设定不同的vmin和vmax
    if label.strip() == "10 hPa":
        vmin, vmax = 0, 1000
    elif label.strip() == "50 hPa":
        vmin, vmax = 0, 100
    else:
        vmin, vmax = None, None
    
    n_frames = min(pv_data1.time.size, pv_data2.time.size)
    for i in range(n_frames):
        pv1 = pv_data1.isel(time=i)
        pv2 = pv_data2.isel(time=i)
        pv1_vals, lons = add_cyclic_point(pv1.values, coord=pv1.lon.values)
        pv2_vals, _ = add_cyclic_point(pv2.values, coord=pv2.lon.values)
        
        time_val = pv1.time.values
        if isinstance(time_val, np.ndarray) and time_val.size == 1:
            time_val = time_val.item()
        time_val = np.datetime64(time_val)
        time_label = np.datetime_as_string(time_val, unit='D')
        
        fig, axes = plt.subplots(1, 2, subplot_kw={'projection': ccrs.NorthPolarStereo()}, figsize=(16,8))
        for ax in axes:
            ax.set_extent([-180, 180, 60, 90], ccrs.PlateCarree())
            ax.coastlines()
            ax.add_feature(cfeature.LAND, facecolor='lightgray')
            gl = ax.gridlines(draw_labels=True, linestyle='--', color='gray')
            gl.xlabels_top = False
            gl.ylabels_right = False
        levels = np.linspace(vmin, vmax, 21)  # 固定21个等值线
        cf1 = axes[0].contourf(lons, pv1.lat, pv1_vals, levels=levels, cmap='plasma',
                         extend='both', transform=ccrs.PlateCarree())
        axes[0].set_title(f"{exp1_name} PV - {label}\nDate: {time_label}", fontsize=14)
        cf2 = axes[1].contourf(lons, pv2.lat, pv2_vals, levels=levels, cmap='plasma',
                         extend='both', transform=ccrs.PlateCarree())
        axes[1].set_title(f"{exp2_name} PV - {label}\nDate: {time_label}", fontsize=14)
        cbar = fig.colorbar(cf2, ax=axes, orientation='horizontal', fraction=0.05, pad=0.08)
        cbar.set_label(f"Potential Vorticity (PVU)", fontsize=12)
        
        out_fname = os.path.join(output_dir, f"frame_{i+1:03d}.png")
        plt.savefig(out_fname, dpi=150, bbox_inches='tight')
        plt.close(fig)
    files = sorted([f for f in os.listdir(output_dir) if f.endswith('.png')])
    if files:
        images = []
        for fname in files:
            fpath = os.path.join(output_dir, fname)
            try:
                images.append(imageio.imread(fpath))
            except Exception as e:
                continue
        gif_fname = os.path.join(output_dir, f"composite_PV_{label.replace(' ', '_')}.gif")
        imageio.mimsave(gif_fname, images, duration=frame_duration/1000)
    else:
        pass

#########################################
# Main processing loop for isobaric PV analysis
#########################################
for year in years:
    for month in months:
        # 加载数据
        T_nocoup = load_year_nocouple_field_var(year, month, 'T')
        U_nocoup = load_year_nocouple_field_var(year, month, 'U')
        V_nocoup = load_year_nocouple_field_var(year, month, 'V')
        T_small = load_year_small_pert_field_var(year, month, 'T')
        U_small = load_year_small_pert_field_var(year, month, 'U')
        V_small = load_year_small_pert_field_var(year, month, 'V')
        
        # 检查并更新纬度坐标：如果最大纬度为90，则将其插值到89
        if np.isclose(T_nocoup.lat.max().values, 90):
            new_lat = T_nocoup.lat.values.copy()
            new_lat[-1] = 89.0
            T_nocoup = T_nocoup.interp(lat=new_lat)
            U_nocoup = U_nocoup.interp(lat=new_lat)
            V_nocoup = V_nocoup.interp(lat=new_lat)
        if np.isclose(T_small.lat.max().values, 90):
            new_lat = T_small.lat.values.copy()
            new_lat[-1] = 89.0
            T_small = T_small.interp(lat=new_lat)
            U_small = U_small.interp(lat=new_lat)
            V_small = V_small.interp(lat=new_lat)
        
        for p in pv_pressure_levels:
            target_pressure = p * 100  # Convert hPa to Pa
            # Select full vertical level
            T_nocoup_level = T_nocoup.sel(plev=slice(None))
            U_nocoup_level = U_nocoup.sel(plev=slice(None))
            V_nocoup_level = V_nocoup.sel(plev=slice(None))
            T_small_level = T_small.sel(plev=slice(None))
            U_small_level = U_small.sel(plev=slice(None))
            V_small_level = V_small.sel(plev=slice(None))
            
            n_time = min(T_nocoup_level.time.size, T_small_level.time.size)
            pv_nocoup_list = []
            pv_small_list = []
            for t in range(n_time):
                T_slice_n = T_nocoup_level.isel(time=t).values
                U_slice_n = U_nocoup_level.isel(time=t).values
                V_slice_n = V_nocoup_level.isel(time=t).values
                
                T_slice_s = T_small_level.isel(time=t).values
                U_slice_s = U_small_level.isel(time=t).values
                V_slice_s = V_small_level.isel(time=t).values
                
                plev_vals = T_nocoup_level.plev.values
                lat_vals = T_nocoup_level.lat.values
                lon_vals = T_nocoup_level.lon.values
                
                try:
                    pv_n = compute_pv_pressure_full(T_slice_n, U_slice_n, V_slice_n, plev_vals, lat_vals, lon_vals, target_pressure)
                    pv_s = compute_pv_pressure_full(T_slice_s, U_slice_s, V_slice_s, plev_vals, lat_vals, lon_vals, target_pressure)
                except Exception as e:
                    continue
                pv_nocoup_list.append(pv_n)
                pv_small_list.append(pv_s)
            
            pv_nocoup_da = xr.DataArray(np.array(pv_nocoup_list),
                                          coords=[T_nocoup_level.time[:n_time], T_nocoup_level.lat, T_nocoup_level.lon],
                                          dims=['time', 'lat', 'lon'])
            pv_small_da = xr.DataArray(np.array(pv_small_list),
                                         coords=[T_small_level.time[:n_time], T_small_level.lat, T_small_level.lon],
                                         dims=['time', 'lat', 'lon'])
            
            # 对计算后的PV场进行后处理，扩展到极点（90°）
            pv_nocoup_da = extend_to_pole(pv_nocoup_da)
            pv_small_da = extend_to_pole(pv_small_da)
            
            out_dir = os.path.join(output_parent, f"{year}_{month}_{p}hPa")
            os.makedirs(out_dir, exist_ok=True)
            plot_composite_daily_maps_PV("nocoupl", pv_nocoup_da, "small", pv_small_da, out_dir, f"{p} hPa", frame_duration=500)

print("All composite GIFs for PV have been generated!")
